In [ ]:
from google.colab import drive

# Google Drive'ı mount et
print("Google Drive mount ediliyor...")
drive.mount('/content/drive')
print("Mount işlemi tamamlandı!")

Google Drive mount ediliyor...
Mounted at /content/drive
Mount işlemi tamamlandı!


In [ ]:
# =========================================
# L1 ONLY PREPROCESSING (EMBEDDED: img + title)
# - Group-aware stratified split
# - Single LabelEncoder
# - Detailed console report
# =========================================
import os, json, re, sys, warnings, textwrap
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter, defaultdict

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight

# --------------- CONFIG ---------------
FILE_PATH = "/content/drive/MyDrive/final_master_df_title_embedded.parquet"
OUT_DIR   = "/content/drive/MyDrive/preproc_l1"

LABEL_COL = "l1"
ID_COL    = "id"       # mevcut dataset sütunu
IMG_COL   = "img_emb"  # 768-dim (object -> list/ndarray)
TTL_COL   = "title"    # 768-dim (object -> list/ndarray)

TARGET_DIM = 768
MIN_SAMPLES_PER_CLASS = 30   # L1 sınıf eşiği
TEST_SIZE = 0.2              # yaklaşık val oranı (SGKF ile 1 fold)
N_SPLITS  = int(round(1/TEST_SIZE))  # SGKF fold sayısı (örn. 0.2 -> 5)
SEED = 42

# Opsiyonel hafif tavan (çok baskın sınıflar için) - None bırakabilirsiniz
GENTLE_MAX_PER_CLASS = None  # örn. 6000 yazarsanız sınıf başına 6000'i kaplar


# --------------- UTILS ---------------
def _print_header(msg):
    print("\n" + "="*70)
    print(msg)
    print("="*70)

def _norm_pathlike_id(s):
    """id içinden dosya adı kökü: klasörleri ve uzantıyı at."""
    s = str(s)
    # yol parçası
    if "/" in s:
        s = s.split("/")[-1]
    if "\\" in s:
        s = s.split("\\")[-1]
    # uzantı
    s = re.sub(r"\.\w+$", "", s)
    return s

def _group_key_from_id(raw_id):
    """
    Heuristik:
    1) path/uzantı temizle
    2) '_' varsa ilk '_' öncesi kök
    3) '-' varsa ilk '-' öncesi kök
    4) değilse komple kök
    """
    rid = _norm_pathlike_id(raw_id)
    if "_" in rid:
        return rid.split("_")[0]
    if "-" in rid:
        return rid.split("-")[0]
    return rid

def _coerce_vec(x, name, row_idx):
    """Liste/ndarray -> np.float32, şekil kontrolü."""
    arr = np.asarray(x, dtype=np.float32)
    if arr.ndim != 1 or arr.shape[0] != TARGET_DIM:
        raise ValueError(f"[Row {row_idx}] {name} bad shape {arr.shape}, expected ({TARGET_DIM},)")
    return arr

def _summarize_class_counts(series, top_k=10):
    vc = series.value_counts()
    total = int(vc.sum())
    uniq  = int(vc.shape[0])
    print(f"- Sınıf sayısı: {uniq}")
    print(f"- Toplam örnek: {total}")
    print(f"- En büyük {top_k} sınıf:")
    for cls, cnt in vc.head(top_k).items():
        print(f"    {cls:>30s} : {cnt}")
    print(f"- En küçük {top_k} sınıf:")
    for cls, cnt in vc.tail(top_k).items():
        print(f"    {cls:>30s} : {cnt}")
    return vc

def _describe_groups(groups):
    # grup boyutları
    gp = pd.Series(groups).value_counts()
    print(f"- Grup sayısı: {gp.shape[0]}")
    qs = gp.quantile([0, .25, .5, .75, .9, .95, 1.0]).to_dict()
    print("- Grup boyutu quantile'ları:", {k: int(v) for k, v in qs.items()})
    if gp.max() == 1:
        print("⚠️  Tüm gruplar tekil görünüyor; varyant ayrımı yakalanmamış olabilir.")

def _balanced_cap(df, label_col, cap=None):
    if not cap:
        return df, {}
    report = {}
    dfs = []
    for c, cdf in df.groupby(label_col):
        if len(cdf) > cap:
            dfs.append(cdf.sample(n=cap, random_state=SEED))
            report[c] = (len(cdf), cap)
        else:
            dfs.append(cdf)
    out = pd.concat(dfs, ignore_index=True)
    return out, report


# --------------- MAIN PIPELINE ---------------
def main():
    os.makedirs(OUT_DIR, exist_ok=True)

    _print_header("1) Veri Yükleme ve Sütun Kontrolü")
    if not Path(FILE_PATH).exists():
        raise FileNotFoundError(FILE_PATH)
    df = pd.read_parquet(FILE_PATH)
    print(f"Dosya: {FILE_PATH}")
    print(f"Satır sayısı: {len(df):,}")

    required_cols = [ID_COL, IMG_COL, TTL_COL, LABEL_COL]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Eksik sütunlar: {missing}")
    print("Sütunlar OK:", required_cols)

    _print_header("2) NaN / Geçersiz Kayıt Temizliği")
    before = len(df)
    df = df.dropna(subset=[IMG_COL, TTL_COL, LABEL_COL])
    dropped_nan = before - len(df)
    print(f"- NaN temizliği: {dropped_nan} satır çıkarıldı")

    # Embedding şekil/doğruluk kontrolü
    bad_rows = []
    for i, (ie, te) in enumerate(zip(df[IMG_COL].values, df[TTL_COL].values)):
        try:
            _ = _coerce_vec(ie, IMG_COL, i)
            _ = _coerce_vec(te, TTL_COL, i)
        except Exception as e:
            bad_rows.append(i)
    if bad_rows:
        print(f"- Geçersiz embedding şekli: {len(bad_rows)} satır çıkarılacak")
        df = df.drop(df.index[bad_rows]).reset_index(drop=True)
    print(f"- Kalan satır: {len(df):,}")

    _print_header("3) L1 Sınıf Dağılımı (Ham)")
    vc_before = _summarize_class_counts(df[LABEL_COL], top_k=10)

    _print_header("4) Min Örnek Eşiği Uygula")
    keep_classes = vc_before[vc_before >= MIN_SAMPLES_PER_CLASS].index
    removed = set(vc_before.index) - set(keep_classes)
    df = df[df[LABEL_COL].isin(keep_classes)].reset_index(drop=True)
    print(f"- Eşik: {MIN_SAMPLES_PER_CLASS}")
    print(f"- Çıkarılan sınıf sayısı: {len(removed)}")
    if removed:
        print("  (Örn. ilk 10)", list(sorted(removed))[:10])
    print(f"- Kalan satır: {len(df):,} | Kalan sınıf: {df[LABEL_COL].nunique()}")

    if GENTLE_MAX_PER_CLASS:
        _print_header("4b) Baskın Sınıflara Hafif Tavan")
        df, cap_report = _balanced_cap(df, LABEL_COL, cap=GENTLE_MAX_PER_CLASS)
        if cap_report:
            print(f"- Kap uygulanan sınıflar: {len(cap_report)}")
            for k, (orig, newn) in list(cap_report.items())[:10]:
                print(f"   {k}: {orig} -> {newn}")
        print(f"- Son satır: {len(df):,}")

    _print_header("5) Grup Anahtarı Çıkarma (Leakage Önleme)")
    # id'den kök grubu üret
    df["_group_key"] = df[ID_COL].map(_group_key_from_id)
    _describe_groups(df["_group_key"])

    _print_header("6) Tek LabelEncoder ile Etiket Haritası")
    le = LabelEncoder()
    le.fit(sorted(df[LABEL_COL].unique()))
    y_all = le.transform(df[LABEL_COL])
    print(f"- LabelEncoder sınıf sayısı: {len(le.classes_)}")
    print(f"- Örnek: {le.classes_[:10]} ...")

    _print_header("7) Embedding Matrislerini Üret (img / title / combined)")
    # Vektörleri güvenle istifle
    def stack_column(col_name):
        return np.stack([_coerce_vec(v, col_name, i) for i, v in enumerate(df[col_name].values)], axis=0)

    X_img   = stack_column(IMG_COL)   # [N, 768]
    X_title = stack_column(TTL_COL)   # [N, 768]
    X_comb  = np.concatenate([X_img, X_title], axis=1).astype(np.float32)  # [N, 1536]

    print(f"- X_img   shape: {X_img.shape}")
    print(f"- X_title shape: {X_title.shape}")
    print(f"- X_comb  shape: {X_comb.shape}")
    print(f"- y_all   shape: {y_all.shape}")

    _print_header("8) StratifiedGroupKFold ile Train/Val Ayrımı")
    # SGKF için kat sayısı (örn. 5) -> ilk fold val, kalanlar train gibi davranacağız
    sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    # SGKF .split(X, y, groups): X kullanılmıyor (emb gerekmiyor)
    splits = list(sgkf.split(np.zeros(len(df)), y_all, df["_group_key"]))
    (train_idx, val_idx) = splits[0]  # ilk fold'u kullan

    # Dağılım raporu
    def cls_dist(y):
        c = Counter(y)
        # en büyük/ küçük 5 sınıf
        big5 = sorted(c.items(), key=lambda z: z[1], reverse=True)[:5]
        small5 = sorted(c.items(), key=lambda z: z[1])[:5]
        return c, big5, small5

    y_tr, y_va = y_all[train_idx], y_all[val_idx]
    ct_tr, b_tr, s_tr = cls_dist(y_tr)
    ct_va, b_va, s_va = cls_dist(y_va)

    print(f"- Train idx: {len(train_idx):,} | Val idx: {len(val_idx):,}")
    print(f"- Train sınıf sayısı: {len(ct_tr)} | Val sınıf sayısı: {len(ct_va)}")
    print("- Train en büyük 5 sınıf:", b_tr)
    print("- Val   en büyük 5 sınıf:", b_va)
    print("- Train en küçük 5 sınıf:", s_tr)
    print("- Val   en küçük 5 sınıf:", s_va)

    # SGKF yine de herhangi bir sınıf val'de 0 olabilir mi? (genelde olmaz ama kontrol)
    missing_in_val = [cls for cls in np.unique(y_all) if cls not in ct_va]
    if missing_in_val:
        print(f"⚠️  Val'de görünmeyen sınıflar: {len(missing_in_val)} -> {missing_in_val[:10]}")

    _print_header("9) Numpy Çıktıları ve Metadata Kaydetme")
    X_img_tr, X_img_va     = X_img[train_idx],   X_img[val_idx]
    X_ttl_tr, X_ttl_va     = X_title[train_idx], X_title[val_idx]
    X_comb_tr, X_comb_va   = X_comb[train_idx],  X_comb[val_idx]
    y_tr, y_va             = y_all[train_idx],   y_all[val_idx]
    groups_tr, groups_va   = df["_group_key"].iloc[train_idx].tolist(), df["_group_key"].iloc[val_idx].tolist()

    # Class weights (train'e göre)
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(len(le.classes_)),
        y=y_tr
    ).astype(np.float32)

    Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
    np.save(f"{OUT_DIR}/X_img_train.npy",   X_img_tr)
    np.save(f"{OUT_DIR}/X_img_val.npy",     X_img_va)
    np.save(f"{OUT_DIR}/X_title_train.npy", X_ttl_tr)
    np.save(f"{OUT_DIR}/X_title_val.npy",   X_ttl_va)
    np.save(f"{OUT_DIR}/X_comb_train.npy",  X_comb_tr)
    np.save(f"{OUT_DIR}/X_comb_val.npy",    X_comb_va)
    np.save(f"{OUT_DIR}/y_train.npy",       y_tr)
    np.save(f"{OUT_DIR}/y_val.npy",         y_va)
    np.save(f"{OUT_DIR}/class_weights.npy", class_weights)

    # label encoder pkl
    import pickle
    with open(f"{OUT_DIR}/label_encoder.pkl", "wb") as f:
        pickle.dump(le, f)

    # metadata (özet)
    meta = {
        "file_path": FILE_PATH,
        "rows_after_clean": int(len(df)),
        "n_classes": int(len(le.classes_)),
        "min_samples_per_class": MIN_SAMPLES_PER_CLASS,
        "gentle_max_per_class": GENTLE_MAX_PER_CLASS,
        "seed": SEED,
        "n_splits": N_SPLITS,
        "split_sizes": {"train": int(len(train_idx)), "val": int(len(val_idx))},
        "shapes": {
            "X_img_train": list(X_img_tr.shape),
            "X_img_val": list(X_img_va.shape),
            "X_title_train": list(X_ttl_tr.shape),
            "X_title_val": list(X_ttl_va.shape),
            "X_comb_train": list(X_comb_tr.shape),
            "X_comb_val": list(X_comb_va.shape),
            "y_train": list(y_tr.shape),
            "y_val": list(y_va.shape),
        },
        "class_weights_summary": {
            "min": float(class_weights.min()),
            "max": float(class_weights.max()),
            "mean": float(class_weights.mean())
        }
    }

    with open(f"{OUT_DIR}/meta.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    # sınıf dağılım tabloları
    train_counts = pd.Series(y_tr).value_counts().sort_index()
    val_counts   = pd.Series(y_va).value_counts().sort_index()
    cls_table = pd.DataFrame({
        "class_idx": np.arange(len(le.classes_)),
        "class_name": le.classes_,
        "train_count": train_counts.reindex(np.arange(len(le.classes_)), fill_value=0).values,
        "val_count": val_counts.reindex(np.arange(len(le.classes_)), fill_value=0).values
    })
    cls_table.to_csv(f"{OUT_DIR}/class_distribution.csv", index=False, encoding="utf-8")

    print("Kaydedilen dosyalar:")
    for fn in [
        "X_img_train.npy","X_img_val.npy",
        "X_title_train.npy","X_title_val.npy",
        "X_comb_train.npy","X_comb_val.npy",
        "y_train.npy","y_val.npy",
        "class_weights.npy","label_encoder.pkl",
        "class_distribution.csv","meta.json"
    ]:
        print(" -", f"{OUT_DIR}/{fn}")

    _print_header("10) Özet")
    print(f"Train: {X_comb_tr.shape[0]:,} | Val: {X_comb_va.shape[0]:,} | Classes: {len(le.classes_)}")
    print("✅ Tek LabelEncoder ile tutarlı etiket eşlemesi")
    print("✅ Group-aware stratified split ile sızıntı riski azaltıldı")
    print("✅ Hem ayrı (img/title) hem birleşik (1536-dim) matrisler kaydedildi")
    print("✅ class_distribution.csv ve meta.json ile detaylı rapor hazır")


if __name__ == "__main__":
    main()



1) Veri Yükleme ve Sütun Kontrolü
Dosya: /content/drive/MyDrive/final_master_df_title_embedded.parquet
Satır sayısı: 72,221
Sütunlar OK: ['id', 'img_emb', 'title', 'l1']

2) NaN / Geçersiz Kayıt Temizliği
- NaN temizliği: 0 satır çıkarıldı
- Kalan satır: 72,221

3) L1 Sınıf Dağılımı (Ham)
- Sınıf sayısı: 84
- Toplam örnek: 72221
- En büyük 10 sınıf:
               Yapı Market & Bahçe : 9869
             Evcil Hayvan Ürünleri : 4416
                       Süpermarket : 2858
            Kadın Giyim & Aksesuar : 2803
                        Bilgisayar : 2573
                  Kırtasiye & Ofis : 2548
                 Aksesuar & Tuning : 2307
                  Mutfak Gereçleri : 2299
                             Kitap : 2289
         Bireysel & Takım Sporları : 1960
- En küçük 10 sınıf:
                 Bebek Oyuncakları : 99
          Dijital Kodlar & Ürünler : 93
          Oto Koltuğu & Ana Kucağı : 80
         Güneş & Bronzluk Ürünleri : 72
         Bebek Bezi & Islak Mendil : 59
     Y

In [ ]:
# =========================================
# L1 MULTIMODAL (img_emb + title_emb) TRAINING
# - Uses pre-embedded 768-d image & 768-d title vectors
# - Group-aware stratified split (leakage korumalı)
# - Single LabelEncoder (train/val/inference tutarlı)
# - WeightedRandomSampler (batch içi dengeleme)
# - Cosine (scaled) classifier head
# - LabelSmoothing + Focal Loss
# - Deterministic training
# =========================================
import os, re, json, warnings, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, top_k_accuracy_score, classification_report, confusion_matrix

from tqdm import tqdm

# ---------------- CONFIG ----------------
FILE_PATH = "/content/drive/MyDrive/final_master_df_title_embedded.parquet"
CKPT_PATH = "/content/drive/MyDrive/best_l1_multimodal_cosine.pth"
REPORT_DIR = "/content/drive/MyDrive/l1_training_reports"

LABEL_COL = "l1"
ID_COL    = "id"
IMG_COL   = "img_emb"   # 768-d
TTL_COL   = "title"     # 768-d (önceden embedlenmiş)

EMB_DIM   = 768
FUSED_DIM = 512
MIN_SAMPLES_PER_CLASS = 30

VAL_RATIO = 0.2
N_SPLITS  = int(round(1/VAL_RATIO))  # 5
SEED      = 42

BATCH_TRAIN = 128
BATCH_VAL   = 256
EPOCHS      = 40
MAX_LR_NEW  = 1e-4   # yeni katmanlar
MAX_LR_BASE = 5e-5   # (burada base yok ama param group tek olacak)

LABEL_SMOOTHING = 0.1
FOCAL_ALPHA     = 1.0
FOCAL_GAMMA     = 2.0

os.makedirs(REPORT_DIR, exist_ok=True)

# -------------- UTILS --------------
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def _norm_pathlike_id(s):
    s = str(s)
    if "/" in s:  s = s.split("/")[-1]
    if "\\" in s: s = s.split("\\")[-1]
    s = re.sub(r"\.\w+$", "", s)
    return s

def group_key_from_id(raw_id):
    rid = _norm_pathlike_id(raw_id)
    if "_" in rid: return rid.split("_")[0]
    if "-" in rid: return rid.split("-")[0]
    return rid

def coerce_vec(x, name, idx):
    arr = np.asarray(x, dtype=np.float32)
    if arr.ndim != 1 or arr.shape[0] != EMB_DIM:
        raise ValueError(f"[Row {idx}] {name} shape {arr.shape}, expected ({EMB_DIM},)")
    return arr

def summarize_split(y_tr, y_va, le):
    def topbot(y):
        c = Counter(y)
        big5 = sorted(c.items(), key=lambda z: z[1], reverse=True)[:5]
        small5 = sorted(c.items(), key=lambda z: z[1])[:5]
        return big5, small5
    btr, strr = topbot(y_tr)
    bva, sva  = topbot(y_va)
    print(f"- Train sınıf sayısı: {len(set(y_tr))} | Val sınıf sayısı: {len(set(y_va))}")
    print("- Train en büyük 5:", [(int(k), v, le.classes_[int(k)]) for k, v in btr])
    print("- Val   en büyük 5:", [(int(k), v, le.classes_[int(k)]) for k, v in bva])
    print("- Train en küçük 5:", [(int(k), v, le.classes_[int(k)]) for k, v in strr])
    print("- Val   en küçük 5:", [(int(k), v, le.classes_[int(k)]) for k, v in sva])

# -------------- DATASET --------------
class EmbeddedEcomDataset(Dataset):
    def __init__(self, X_img, X_ttl, y):
        self.X_img = X_img.astype(np.float32)
        self.X_ttl = X_ttl.astype(np.float32)
        self.y     = y.astype(np.int64)

    def __len__(self): return self.X_img.shape[0]

    def __getitem__(self, idx):
        return {
            "image_features": torch.from_numpy(self.X_img[idx]),
            "title_features": torch.from_numpy(self.X_ttl[idx]),
            "labels": torch.tensor(self.y[idx], dtype=torch.long)
        }

# -------------- MODEL --------------
class ClassTokenAttention(nn.Module):
    def __init__(self, feature_dim, num_classes):
        super().__init__()
        self.feature_dim = feature_dim
        self.class_prototypes = nn.Parameter(torch.randn(num_classes, feature_dim))
        self.q = nn.Linear(feature_dim, feature_dim)
        self.k = nn.Linear(feature_dim, feature_dim)
        self.v = nn.Linear(feature_dim, feature_dim)
        self.drop = nn.Dropout(0.1)
        self.out = nn.Linear(feature_dim, feature_dim)
        nn.init.xavier_uniform_(self.class_prototypes)

    def forward(self, x):
        # x: [B, D]
        q = self.q(x)                                # [B, D]
        k = self.k(self.class_prototypes)            # [C, D]
        v = self.v(self.class_prototypes)            # [C, D]
        att = torch.matmul(q, k.t()) / (x.shape[1] ** 0.5)  # [B, C]
        w = F.softmax(att, dim=1)
        w = self.drop(w)
        attended = torch.matmul(w, v)                # [B, D]
        out = self.out(attended + x)                 # residual
        return out, w

class CosineClassifier(nn.Module):
    def __init__(self, in_dim, num_classes, scale=30.0):
        super().__init__()
        self.scale = scale
        self.weight = nn.Parameter(torch.randn(num_classes, in_dim))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, feats):
        w = F.normalize(self.weight, dim=1)  # [C, D]
        f = F.normalize(feats, dim=1)        # [B, D]
        logits = self.scale * torch.matmul(f, w.t())
        return logits

class MultiModalL1(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.fusion = nn.Sequential(
            nn.Linear(EMB_DIM*2, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(1024, FUSED_DIM),
            nn.LayerNorm(FUSED_DIM),
            nn.GELU(),
            nn.Dropout(0.2),
        )
        self.attn = ClassTokenAttention(FUSED_DIM, num_classes)
        self.head = CosineClassifier(FUSED_DIM, num_classes, scale=30.0)

        # init
        for m in self.fusion.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.LayerNorm):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, img_feats, ttl_feats):
        img_feats = F.normalize(img_feats, dim=-1)
        ttl_feats = F.normalize(ttl_feats, dim=-1)
        x = torch.cat([img_feats, ttl_feats], dim=1)  # [B, 1536]
        x = self.fusion(x)                            # [B, 512]
        x, attw = self.attn(x)                        # [B, 512], [B, C]
        logits = self.head(x)                         # [B, C]
        return logits, attw

# -------------- LOSSES --------------
class LabelSmoothingFocalLoss(nn.Module):
    def __init__(self, num_classes, smoothing=0.1, alpha=1.0, gamma=2.0, weight=None):
        super().__init__()
        self.num_classes = num_classes
        self.smoothing = smoothing
        self.alpha = alpha
        self.gamma = gamma
        self.register_buffer("weight", weight if weight is not None else None)

    def forward(self, inputs, targets):
        # label smoothing
        with torch.no_grad():
            confidence = 1.0 - self.smoothing
            smooth_neg = self.smoothing / (self.num_classes - 1)
            smoothed = torch.full_like(inputs, smooth_neg)
            smoothed.scatter_(1, targets.unsqueeze(1), confidence)

        logp = F.log_softmax(inputs, dim=1)
        pt = torch.exp(logp.gather(1, targets.unsqueeze(1)).squeeze(1))  # [B]
        focal = self.alpha * (1 - pt) ** self.gamma

        loss = -(smoothed * logp).sum(dim=1) * focal  # [B]
        if self.weight is not None:
            loss = loss * self.weight[targets]
        return loss.mean()

# -------------- TRAINER --------------
class Trainer:
    def __init__(self, model, device, train_loader, val_loader, num_classes, class_weights, label_classes):
        self.model = model.to(device)
        self.device = device
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.num_classes = num_classes
        self.label_classes = label_classes

        cw = torch.tensor(class_weights, dtype=torch.float32, device=device) if class_weights is not None else None
        self.criterion = LabelSmoothingFocalLoss(num_classes, LABEL_SMOOTHING, FOCAL_ALPHA, FOCAL_GAMMA, cw)

        # tek param grubu (fusion+attn+head)
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=MAX_LR_NEW, weight_decay=1e-3)

        self.scheduler = torch.optim.lr_scheduler.OneCycleLR(
            self.optimizer,
            max_lr=MAX_LR_NEW,
            epochs=EPOCHS,
            steps_per_epoch=len(self.train_loader),
            pct_start=0.1,
            anneal_strategy='cos',
            div_factor=10,
            final_div_factor=100
        )

        self.scaler = torch.cuda.amp.GradScaler()
        self.best_val = 0.0

    def train_epoch(self):
        self.model.train()
        tot_loss, correct, total = 0.0, 0, 0
        pbar = tqdm(self.train_loader, desc="Train")
        for batch in pbar:
            xi = batch["image_features"].to(self.device)
            xt = batch["title_features"].to(self.device)
            y  = batch["labels"].to(self.device)

            self.optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast():
                logits, _ = self.model(xi, xt)
                loss = self.criterion(logits, y)

            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.scaler.step(self.optimizer)
            self.scaler.update()
            self.scheduler.step()

            tot_loss += loss.item()
            pred = logits.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{100*correct/total:.2f}%")
        return tot_loss/len(self.train_loader), 100*correct/total

    @torch.no_grad()
    def validate(self):
        self.model.eval()
        tot_loss = 0.0
        all_logits, all_labels = [], []
        pbar = tqdm(self.val_loader, desc="Val")
        for batch in pbar:
            xi = batch["image_features"].to(self.device)
            xt = batch["title_features"].to(self.device)
            y  = batch["labels"].to(self.device)
            with torch.cuda.amp.autocast():
                logits, _ = self.model(xi, xt)
                loss = self.criterion(logits, y)
            tot_loss += loss.item()
            all_logits.append(logits.float().cpu().numpy())
            all_labels.append(y.cpu().numpy())
        all_logits = np.concatenate(all_logits, 0)
        all_labels = np.concatenate(all_labels, 0)
        top1 = accuracy_score(all_labels, all_logits.argmax(1))*100
        top5 = top_k_accuracy_score(all_labels, all_logits, k=5)*100 if self.num_classes>=5 else top1
        return tot_loss/len(self.val_loader), top1, top5, all_logits, all_labels

    def save_ckpt(self, path, epoch, val_top1, val_top5):
        torch.save({
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "epoch": epoch,
            "val_acc": val_top1,
            "val_top5": val_top5,
            "label_classes": list(self.label_classes)
        }, path)

    def fit(self, epochs=EPOCHS):
        print(f"Starting training for {epochs} epochs")
        best = 0.0
        patience, max_patience = 0, 10
        for ep in range(1, epochs+1):
            print(f"\nEpoch {ep}/{epochs}")
            tr_loss, tr_acc = self.train_epoch()
            va_loss, va_top1, va_top5, logits, labels = self.validate()

            print(f"Train Loss: {tr_loss:.4f} | Train Acc: {tr_acc:.2f}%")
            print(f"Val   Loss: {va_loss:.4f} | Val Top-1: {va_top1:.2f}% | Val Top-5: {va_top5:.2f}%")

            if va_top1 > best:
                best = va_top1
                patience = 0
                self.save_ckpt(CKPT_PATH, ep, va_top1, va_top5)
                print(f"✅ New best checkpoint saved to: {CKPT_PATH} (Top-1 {va_top1:.2f}%)")
                # Rapor (sınıf bazlı)
                preds = logits.argmax(1)
                self._write_reports(labels, preds)
            else:
                patience += 1
                print(f"No improvement. Patience {patience}/10")

            if patience >= max_patience:
                print("⏹️ Early stopping.")
                break

            if va_top1 >= 99.0:
                print("🎯 TARGET ACHIEVED ≥99%")
                break
        print(f"\nBest Val Top-1: {best:.2f}%")
        return best

    def _write_reports(self, y_true, y_pred):
        # sınıf isimleriyle classification report
        try:
            report = classification_report(
                y_true, y_pred,
                target_names=[str(x) for x in self.label_classes],
                digits=4, zero_division=0
            )
            with open(os.path.join(REPORT_DIR, "classification_report.txt"), "w", encoding="utf-8") as f:
                f.write(report)
            print("📄 classification_report.txt yazıldı.")

            cm = confusion_matrix(y_true, y_pred)
            np.save(os.path.join(REPORT_DIR, "confusion_matrix.npy"), cm)
            print("📊 confusion_matrix.npy yazıldı.")
        except Exception as e:
            print("Rapor yazımında hata:", e)

# -------------- DATA PREP --------------
def load_and_prepare():
    print("="*70)
    print("1) Veri Yükleme ve Kolon Kontrolü")
    print("="*70)
    if not Path(FILE_PATH).exists():
        raise FileNotFoundError(FILE_PATH)
    df = pd.read_parquet(FILE_PATH)
    print(f"- Dosya: {FILE_PATH}")
    print(f"- Satır: {len(df):,}")
    for c in [ID_COL, IMG_COL, TTL_COL, LABEL_COL]:
        if c not in df.columns:
            raise KeyError(f"Eksik sütun: {c}")
    print("- Sütunlar OK:", [ID_COL, IMG_COL, TTL_COL, LABEL_COL])

    print("\n" + "="*70)
    print("2) NaN / Embed Şekil Kontrolü")
    print("="*70)
    before = len(df)
    df = df.dropna(subset=[IMG_COL, TTL_COL, LABEL_COL])
    print(f"- NaN temizliği: {before - len(df)} satır çıkarıldı")
    bad = []
    for i, (ie, te) in enumerate(zip(df[IMG_COL].values, df[TTL_COL].values)):
        try:
            _ = coerce_vec(ie, IMG_COL, i)
            _ = coerce_vec(te, TTL_COL, i)
        except Exception:
            bad.append(i)
    if bad:
        print(f"- Embed shape hatalı: {len(bad)} satır çıkarılıyor")
        df = df.drop(df.index[bad]).reset_index(drop=True)
    print(f"- Kalan satır: {len(df):,}")

    print("\n" + "="*70)
    print("3) L1 Sınıf Dağılımı (Ham)")
    print("="*70)
    vc = df[LABEL_COL].value_counts()
    print(f"- Sınıf sayısı: {vc.shape[0]} | Toplam: {vc.sum():,}")
    print("- En büyük 10:", vc.head(10).to_dict())
    print("- En küçük 10:", vc.tail(10).to_dict())

    print("\n" + "="*70)
    print("4) Min Örnek Eşiği")
    print("="*70)
    keep = vc[vc >= MIN_SAMPLES_PER_CLASS].index
    removed = set(vc.index) - set(keep)
    df = df[df[LABEL_COL].isin(keep)].reset_index(drop=True)
    print(f"- Eşik: {MIN_SAMPLES_PER_CLASS}")
    print(f"- Çıkarılan sınıf: {len(removed)} -> örn: {list(sorted(removed))[:10]}")
    print(f"- Kalan satır: {len(df):,} | Kalan sınıf: {df[LABEL_COL].nunique()}")

    print("\n" + "="*70)
    print("5) Grup Anahtarı (Leakage Önleme)")
    print("="*70)
    df["_group_key"] = df[ID_COL].map(group_key_from_id)
    gp_counts = df["_group_key"].value_counts()
    quant = gp_counts.quantile([0, .25, .5, .75, .9, .95, 1.0]).to_dict()
    print(f"- Grup sayısı: {gp_counts.shape[0]}")
    print("- Grup boyutu quantile'ları:", {k: int(v) for k, v in quant.items()})
    if gp_counts.max() == 1:
        print("⚠️  Tüm gruplar tekil görünüyor (yine de SGKF güvenli).")

    print("\n" + "="*70)
    print("6) Tek LabelEncoder ve Embedding Matrisleri")
    print("="*70)
    le = LabelEncoder()
    le.fit(sorted(df[LABEL_COL].unique()))
    y_all = le.transform(df[LABEL_COL])
    print(f"- Sınıf sayısı (LE): {len(le.classes_)}")
    print(f"- İlk 10 sınıf: {list(le.classes_)[:10]}")

    X_img = np.stack([coerce_vec(v, IMG_COL, i) for i, v in enumerate(df[IMG_COL].values)], axis=0)
    X_ttl = np.stack([coerce_vec(v, TTL_COL, i) for i, v in enumerate(df[TTL_COL].values)], axis=0)
    print(f"- X_img shape   : {X_img.shape}")
    print(f"- X_title shape : {X_ttl.shape}")
    print(f"- y_all shape   : {y_all.shape}")

    print("\n" + "="*70)
    print("7) StratifiedGroupKFold Split")
    print("="*70)
    sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    splits = list(sgkf.split(np.zeros(len(df)), y_all, df["_group_key"]))
    train_idx, val_idx = splits[0]
    print(f"- Train idx: {len(train_idx):,} | Val idx: {len(val_idx):,}")

    X_img_tr, X_img_va = X_img[train_idx], X_img[val_idx]
    X_ttl_tr, X_ttl_va = X_ttl[train_idx], X_ttl[val_idx]
    y_tr, y_va = y_all[train_idx], y_all[val_idx]

    summarize_split(y_tr, y_va, le)

    # class weights
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(len(le.classes_)),
        y=y_tr
    ).astype(np.float32)
    print(f"- Class weights: min={class_weights.min():.4f} max={class_weights.max():.4f} mean={class_weights.mean():.4f}")

    return (X_img_tr, X_ttl_tr, y_tr), (X_img_va, X_ttl_va, y_va), le, class_weights

# -------------- MAIN --------------
def main():
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("="*70)
    print("L1 Training (Pre-embedded img + title)")
    print("Device:", device)
    print("="*70)

    (Xi_tr, Xt_tr, y_tr), (Xi_va, Xt_va, y_va), le, class_weights = load_and_prepare()
    num_classes = len(le.classes_)

    # Datasets
    train_ds = EmbeddedEcomDataset(Xi_tr, Xt_tr, y_tr)
    val_ds   = EmbeddedEcomDataset(Xi_va, Xt_va, y_va)

    # Weighted sampler
    binc = np.bincount(y_tr, minlength=num_classes)
    binc[binc == 0] = 1
    weights = 1.0 / binc
    sample_weights = weights[y_tr]
    sampler = WeightedRandomSampler(torch.DoubleTensor(sample_weights),
                                    num_samples=len(sample_weights),
                                    replacement=True)

    # Loaders
    train_loader = DataLoader(train_ds, batch_size=BATCH_TRAIN, sampler=sampler,
                              num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_VAL, shuffle=False,
                              num_workers=4, pin_memory=True)

    # Model
    model = MultiModalL1(num_classes=num_classes)

    # Trainer
    trainer = Trainer(model, device, train_loader, val_loader, num_classes, class_weights, le.classes_)
    best = trainer.fit(EPOCHS)

    print("\nFINAL RESULT (Best Val Top-1): {:.2f}%".format(best))
    if best < 99.0:
        print("Still {:.2f}% away from 99% target".format(99.0 - best))

if __name__ == "__main__":
    main()


L1 Training (Pre-embedded img + title)
Device: cuda
1) Veri Yükleme ve Kolon Kontrolü
- Dosya: /content/drive/MyDrive/final_master_df_title_embedded.parquet
- Satır: 72,221
- Sütunlar OK: ['id', 'img_emb', 'title', 'l1']

2) NaN / Embed Şekil Kontrolü
- NaN temizliği: 0 satır çıkarıldı
- Kalan satır: 72,221

3) L1 Sınıf Dağılımı (Ham)
- Sınıf sayısı: 84 | Toplam: 72,221
- En büyük 10: {'Yapı Market & Bahçe': 9869, 'Evcil Hayvan Ürünleri': 4416, 'Süpermarket': 2858, 'Kadın Giyim & Aksesuar': 2803, 'Bilgisayar': 2573, 'Kırtasiye & Ofis': 2548, 'Aksesuar & Tuning': 2307, 'Mutfak Gereçleri': 2299, 'Kitap': 2289, 'Bireysel & Takım Sporları': 1960}
- En küçük 10: {'Bebek Oyuncakları': 99, 'Dijital Kodlar & Ürünler': 93, 'Oto Koltuğu & Ana Kucağı': 80, 'Güneş & Bronzluk Ürünleri': 72, 'Bebek Bezi & Islak Mendil': 59, 'Yürüteç & Yürüme Yardımcıları': 40, 'Ev Kozmetiği': 40, 'Yaşam ve Etkinlik': 21, 'El İşi Ürünleri': 20, 'Kampanya Kozmetik': 3}

4) Min Örnek Eşiği
- Eşik: 30
- Çıkarılan sınıf:

Val: 100%|██████████| 57/57 [00:00<00:00, 153.16it/s]


Train Loss: 10.4670 | Train Acc: 12.02%
Val   Loss: 3.4715 | Val Top-1: 5.52% | Val Top-5: 11.19%
✅ New best checkpoint saved to: /content/drive/MyDrive/best_l1_multimodal_cosine.pth (Top-1 5.52%)
📄 classification_report.txt yazıldı.
📊 confusion_matrix.npy yazıldı.

Epoch 2/40


Val: 100%|██████████| 57/57 [00:00<00:00, 173.12it/s]


Train Loss: 4.1412 | Train Acc: 36.98%
Val   Loss: 2.0540 | Val Top-1: 18.93% | Val Top-5: 42.09%
✅ New best checkpoint saved to: /content/drive/MyDrive/best_l1_multimodal_cosine.pth (Top-1 18.93%)
📄 classification_report.txt yazıldı.
📊 confusion_matrix.npy yazıldı.

Epoch 3/40


...

Train Loss: 0.0572 | Train Acc: 95.58%
Val   Loss: 0.9177 | Val Top-1: 72.41% | Val Top-5: 93.23%
✅ New best checkpoint saved to: /content/drive/MyDrive/best_l1_multimodal_cosine.pth (Top-1 72.41%)
📄 classification_report.txt yazıldı.
📊 confusion_matrix.npy yazıldı.

Epoch 39/40


Val: 100%|██████████| 57/57 [00:00<00:00, 152.25it/s]


Train Loss: 0.0575 | Train Acc: 95.52%
Val   Loss: 0.9197 | Val Top-1: 72.34% | Val Top-5: 93.14%
No improvement. Patience 1/10

Epoch 40/40


Val: 100%|██████████| 57/57 [00:00<00:00, 169.55it/s]

Train Loss: 0.0560 | Train Acc: 95.67%
Val   Loss: 0.9190 | Val Top-1: 72.36% | Val Top-5: 93.16%
No improvement. Patience 2/10

Best Val Top-1: 72.41%

FINAL RESULT (Best Val Top-1): 72.41%
Still 26.59% away from 99% target


In [ ]:
!pip install faiss-gpu-cu12


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 106.1 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you

In [ ]:
# =========================================
# L1 FAISS INTEGRATION (Pre-embedded img/title, new model)
# - Builds FAISS over train fused features (512-d)
# - Compares model-only vs FAISS-hybrid on val set
# - Saves index, meta, and evaluation CSV
# =========================================
import os, re, json, warnings, random, pickle
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import faiss
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# ---------- Paths (adjust if needed) ----------
PREPROC_DIR = "/content/drive/MyDrive/preproc_l1"
CKPT_PATH   = "/content/drive/MyDrive/best_l1_multimodal_cosine.pth"

FAISS_INDEX_PATH = "/content/drive/MyDrive/l1_faiss.index"
FAISS_META_PATH  = "/content/drive/MyDrive/l1_faiss_meta.pkl"

COMPARE_CSV_PATH = "/content/drive/MyDrive/eval_l1_faiss_vs_model.csv"

# ---------- Model dims ----------
EMB_DIM   = 768
FUSED_DIM = 512

# ---------- Seed ----------
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ----------------- Model (same as training) -----------------
class ClassTokenAttention(nn.Module):
    def __init__(self, feature_dim, num_classes):
        super().__init__()
        self.feature_dim = feature_dim
        self.class_prototypes = nn.Parameter(torch.randn(num_classes, feature_dim))
        self.q = nn.Linear(feature_dim, feature_dim)
        self.k = nn.Linear(feature_dim, feature_dim)
        self.v = nn.Linear(feature_dim, feature_dim)
        self.drop = nn.Dropout(0.1)
        self.out = nn.Linear(feature_dim, feature_dim)
        nn.init.xavier_uniform_(self.class_prototypes)

    def forward(self, x):
        q = self.q(x)                                # [B, D]
        k = self.k(self.class_prototypes)            # [C, D]
        v = self.v(self.class_prototypes)            # [C, D]
        att = torch.matmul(q, k.t()) / (x.shape[1] ** 0.5)  # [B, C]
        w = F.softmax(att, dim=1)
        w = self.drop(w)
        attended = torch.matmul(w, v)                # [B, D]
        out = self.out(attended + x)                 # residual
        return out, w

class CosineClassifier(nn.Module):
    def __init__(self, in_dim, num_classes, scale=30.0):
        super().__init__()
        self.scale = scale
        self.weight = nn.Parameter(torch.randn(num_classes, in_dim))
        nn.init.xavier_uniform_(self.weight)
    def forward(self, feats):
        w = F.normalize(self.weight, dim=1)  # [C, D]
        f = F.normalize(feats, dim=1)        # [B, D]
        logits = self.scale * torch.matmul(f, w.t())
        return logits

class MultiModalL1(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.fusion = nn.Sequential(
            nn.Linear(EMB_DIM*2, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(1024, FUSED_DIM),
            nn.LayerNorm(FUSED_DIM),
            nn.GELU(),
            nn.Dropout(0.2),
        )
        self.attn = ClassTokenAttention(FUSED_DIM, num_classes)
        self.head = CosineClassifier(FUSED_DIM, num_classes, scale=30.0)
        # init
        for m in self.fusion.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.LayerNorm):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    # helper: extract fused features only
    def fused_features(self, img_feats, ttl_feats):
        img_feats = F.normalize(img_feats, dim=-1)
        ttl_feats = F.normalize(ttl_feats, dim=-1)
        x = torch.cat([img_feats, ttl_feats], dim=1)  # [B, 1536]
        x = self.fusion(x)                            # [B, 512]
        x, _ = self.attn(x)                           # [B, 512]
        return x

    def forward(self, img_feats, ttl_feats):
        x = self.fused_features(img_feats, ttl_feats) # [B, 512]
        logits = self.head(x)                         # [B, C]
        return logits


# ----------------- System -----------------
class L1FAISSSystem:
    def __init__(self,
                 preproc_dir=PREPROC_DIR,
                 ckpt_path=CKPT_PATH,
                 device="cuda"):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.preproc_dir = preproc_dir
        self.ckpt_path = ckpt_path

        self.model = None
        self.label_classes = None
        self.num_classes = None

        # refs for FAISS
        self.faiss_index = None
        self.ref_labels = None  # np.int64 array

        # data arrays
        self.Xi_tr = self.Xt_tr = self.y_tr = None
        self.Xi_va = self.Xt_va = self.y_va = None

        print(f"[System] Device: {self.device}")

    # ---------- data ----------
    def load_preproc_arrays(self):
        print("Loading preprocessed arrays from:", self.preproc_dir)
        Xi_tr = np.load(os.path.join(self.preproc_dir, "X_img_train.npy"))
        Xt_tr = np.load(os.path.join(self.preproc_dir, "X_title_train.npy"))
        y_tr  = np.load(os.path.join(self.preproc_dir, "y_train.npy"))

        Xi_va = np.load(os.path.join(self.preproc_dir, "X_img_val.npy"))
        Xt_va = np.load(os.path.join(self.preproc_dir, "X_title_val.npy"))
        y_va  = np.load(os.path.join(self.preproc_dir, "y_val.npy"))

        # sanity checks
        assert Xi_tr.shape[1] == EMB_DIM and Xt_tr.shape[1] == EMB_DIM
        assert Xi_va.shape[1] == EMB_DIM and Xt_va.shape[1] == EMB_DIM

        self.Xi_tr, self.Xt_tr, self.y_tr = Xi_tr.astype(np.float32), Xt_tr.astype(np.float32), y_tr.astype(np.int64)
        self.Xi_va, self.Xt_va, self.y_va = Xi_va.astype(np.float32), Xt_va.astype(np.float32), y_va.astype(np.int64)

        print(f"Train: Xi {self.Xi_tr.shape}, Xt {self.Xt_tr.shape}, y {self.y_tr.shape}")
        print(f"Val  : Xi {self.Xi_va.shape}, Xt {self.Xt_va.shape}, y {self.y_va.shape}")

    # ---------- model ----------
    def load_model(self):
        print("Loading model checkpoint:", self.ckpt_path)
        ckpt = torch.load(self.ckpt_path, map_location=self.device, weights_only=False)
        # label classes from ckpt
        if "label_classes" in ckpt:
            self.label_classes = np.array(ckpt["label_classes"])
        else:
            # fallback: try label_encoder.pkl from preproc
            le_pkl = os.path.join(self.preproc_dir, "label_encoder.pkl")
            with open(le_pkl, "rb") as f:
                le = pickle.load(f)
            self.label_classes = np.array(le.classes_)
        self.num_classes = int(len(self.label_classes))
        print(f"- Classes: {self.num_classes}")

        self.model = MultiModalL1(self.num_classes).to(self.device).eval()
        self.model.load_state_dict(ckpt["model_state_dict"], strict=True)
        self.model.eval()
        print("✅ Model loaded & ready.")

    # ---------- feature extraction ----------
    @torch.no_grad()
    def _batch_fused(self, Xi, Xt, batch_size=2048):
        self.model.eval()
        feats = []
        for i in range(0, Xi.shape[0], batch_size):
            xi = torch.from_numpy(Xi[i:i+batch_size]).to(self.device)
            xt = torch.from_numpy(Xt[i:i+batch_size]).to(self.device)
            z  = self.model.fused_features(xi, xt)  # [B,512]
            feats.append(z.float().cpu().numpy())
        feats = np.vstack(feats).astype(np.float32)
        return feats  # [N,512]

    @torch.no_grad()
    def _batch_logits(self, Xi, Xt, batch_size=2048):
        self.model.eval()
        out = []
        for i in range(0, Xi.shape[0], batch_size):
            xi = torch.from_numpy(Xi[i:i+batch_size]).to(self.device)
            xt = torch.from_numpy(Xt[i:i+batch_size]).to(self.device)
            logits = self.model(xi, xt)  # [B,C]
            out.append(logits.float().cpu().numpy())
        return np.vstack(out).astype(np.float32)

    # ---------- FAISS build/save/load ----------
    def build_faiss(self, index_type="ivf", nlist=100):
        if self.model is None: self.load_model()
        if self.Xi_tr is None: self.load_preproc_arrays()

        print(f"Building FAISS over train fused features... (type={index_type})")
        feats = self._batch_fused(self.Xi_tr, self.Xt_tr, batch_size=2048)  # [N,512]
        faiss.normalize_L2(feats)
        d = feats.shape[1]

        if index_type == "flat":
            index = faiss.IndexFlatIP(d)
        elif index_type == "ivf":
            quant = faiss.IndexFlatIP(d)
            index = faiss.IndexIVFFlat(quant, d, nlist, faiss.METRIC_INNER_PRODUCT)
            print("Training IVF...")
            index.train(feats)
        elif index_type == "hnsw":
            index = faiss.IndexHNSWFlat(d, 32)
            index.hnsw.efConstruction = 40
        else:
            raise ValueError("index_type must be one of: flat | ivf | hnsw")

        index.add(feats)
        self.faiss_index = index
        self.ref_labels  = self.y_tr.copy()
        print(f"✅ FAISS built: {index.ntotal:,} vectors (d={d})")

        self._save_index()

    def _save_index(self):
        print("Saving FAISS index + meta...")
        faiss.write_index(self.faiss_index, FAISS_INDEX_PATH)
        meta = {
            "classes": self.label_classes,
            "ref_labels": self.ref_labels,
            "train_size": int(self.Xi_tr.shape[0]),
            "dim": int(self.faiss_index.d),
        }
        with open(FAISS_META_PATH, "wb") as f:
            pickle.dump(meta, f)
        print(f"💾 Saved index -> {FAISS_INDEX_PATH}")
        print(f"💾 Saved meta  -> {FAISS_META_PATH}")

    def load_index(self):
        try:
            self.faiss_index = faiss.read_index(FAISS_INDEX_PATH)
            with open(FAISS_META_PATH, "rb") as f:
                meta = pickle.load(f)
            self.label_classes = np.array(meta["classes"])
            self.ref_labels = meta["ref_labels"]
            self.num_classes = int(len(self.label_classes))
            print(f"✅ Loaded FAISS index: {self.faiss_index.ntotal:,} vectors (d={self.faiss_index.d})")
            return True
        except Exception as e:
            print("❌ Could not load FAISS index:", e)
            return False

    # ---------- Predictors ----------
    @torch.no_grad()
    def predict_model_only(self, Xi, Xt, batch_size=2048):
        logits = self._batch_logits(Xi, Xt, batch_size=batch_size)  # [N,C]
        probs  = torch.softmax(torch.from_numpy(logits), dim=1).numpy()
        preds  = probs.argmax(1)
        conf   = probs.max(1)
        return preds, conf, probs

    @torch.no_grad()
    def predict_hybrid(self, Xi, Xt, k=10, alpha=0.6, batch_size=2048):
        if self.faiss_index is None:
            if not self.load_index():
                self.build_faiss()

        # model probs
        logits = self._batch_logits(Xi, Xt, batch_size=batch_size)  # [N,C]
        mprobs = torch.softmax(torch.from_numpy(logits), dim=1).numpy()  # [N,C]

        # query feats for FAISS
        q = self._batch_fused(Xi, Xt, batch_size=batch_size)  # [N,512]
        faiss.normalize_L2(q)
        sims, idxs = self.faiss_index.search(q, k)            # [N,k], [N,k]

        # convert neighbor sims to class distribution
        N, C = q.shape[0], self.num_classes
        fprobs = np.zeros((N, C), dtype=np.float32)
        for i in range(N):
            neigh_idx = idxs[i]
            neigh_sim = sims[i]
            for j in range(k):
                lab = int(self.ref_labels[neigh_idx[j]])
                fprobs[i, lab] += neigh_sim[j]
            s = fprobs[i].sum()
            if s > 0: fprobs[i] /= s

        hyb = alpha * mprobs + (1 - alpha) * fprobs
        preds = hyb.argmax(1)
        conf  = hyb.max(1)
        return preds, conf, hyb

    # ---------- Evaluation / Comparison ----------
    def compare_on_val(self, k=10, alpha=0.6, max_samples=None, save_csv=COMPARE_CSV_PATH):
        if self.model is None: self.load_model()
        if self.Xi_tr is None: self.load_preproc_arrays()
        if self.faiss_index is None:
            if not self.load_index():
                self.build_faiss()

        # sample subset if requested
        Xi, Xt, y = self.Xi_va, self.Xt_va, self.y_va
        if max_samples is not None and max_samples < Xi.shape[0]:
            rs = np.random.RandomState(SEED)
            idx = rs.choice(Xi.shape[0], size=max_samples, replace=False)
            Xi, Xt, y = Xi[idx], Xt[idx], y[idx]

        print(f"Evaluating on val: N={Xi.shape[0]} | k={k} | alpha={alpha}")

        pm, cm, _ = self.predict_model_only(Xi, Xt)
        ph, ch, _ = self.predict_hybrid(Xi, Xt, k=k, alpha=alpha)

        acc_m = accuracy_score(y, pm) * 100.0
        acc_h = accuracy_score(y, ph) * 100.0

        print(f"Model-only  accuracy: {acc_m:.2f}%")
        print(f"Hybrid(FAISS) accuracy: {acc_h:.2f}%  (Δ={acc_h-acc_m:+.2f} pp)")

        # save details
        df = pd.DataFrame({
            "true": self.label_classes[y],
            "pred_model": self.label_classes[pm],
            "pred_hybrid": self.label_classes[ph],
            "conf_model": cm,
            "conf_hybrid": ch
        })
        df.to_csv(save_csv, index=False)
        print(f"📄 Saved details -> {save_csv}")

        return {
            "acc_model": acc_m,
            "acc_hybrid": acc_h,
            "delta_pp": acc_h - acc_m,
            "k": k,
            "alpha": alpha,
            "n_eval": int(Xi.shape[0])
        }


# ----------------- MAIN -----------------
def main():
    set_seed(SEED)
    system = L1FAISSSystem(
        preproc_dir=PREPROC_DIR,
        ckpt_path=CKPT_PATH,
        device="cuda"
    )
    # Load everything & build index if not present
    system.load_model()
    system.load_preproc_arrays()
    if not system.load_index():
        system.build_faiss(index_type="ivf", nlist=100)

    # Compare on validation split
    result = system.compare_on_val(k=10, alpha=0.6, max_samples=5000, save_csv=COMPARE_CSV_PATH)
    print("SUMMARY:", result)

if __name__ == "__main__":
    main()



[System] Device: cuda
Loading model checkpoint: /content/drive/MyDrive/best_l1_multimodal_cosine.pth
- Classes: 81
✅ Model loaded & ready.
Loading preprocessed arrays from: /content/drive/MyDrive/preproc_l1
Train: Xi (57743, 768), Xt (57743, 768), y (57743,)
Val  : Xi (14434, 768), Xt (14434, 768), y (14434,)
❌ Could not load FAISS index: Error in faiss::FileIOReader::FileIOReader(const char*) at /project/faiss/faiss/impl/io.cpp:69: Error: 'f' failed: could not open /content/drive/MyDrive/l1_faiss.index for reading: No such file or directory
Building FAISS over train fused features... (type=ivf)
Training IVF...
✅ FAISS built: 57,743 vectors (d=512)
Saving FAISS index + meta...
💾 Saved index -> /content/drive/MyDrive/l1_faiss.index
💾 Saved meta  -> /content/drive/MyDrive/l1_faiss_meta.pkl
Evaluating on val: N=5000 | k=10 | alpha=0.6
Model-only  accuracy: 72.42%
Hybrid(FAISS) accuracy: 75.42%  (Δ=+3.00 pp)
📄 Saved details -> /content/drive/MyDrive/eval_l1_faiss_vs_model.csv
SUMMARY: {'ac

In [ ]:
# =========================================
# L1 TRAIN (Pre-embedded img/title) — A100 Ready
# - Group-aware stratified split (leakage guard)
# - Single LabelEncoder
# - Gating (text-weighted) + ArcFace head
# - LabelSmoothing + Focal (LS=0.05, gamma=1.5)
# - WeightedRandomSampler + OneCycleLR + AMP
# - Saves best fusion checkpoint & logits
# - Trains Text-only LogisticRegression and ENSEMBLEs with fusion
# =========================================
import os, re, json, warnings, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, top_k_accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
import joblib

from tqdm import tqdm

# ---------------- CONFIG ----------------
FILE_PATH = "/content/drive/MyDrive/final_master_df_title_embedded.parquet"

# Çıkışlar
CKPT_PATH   = "/content/drive/MyDrive/best_l1_multimodal_arcface.pth"
REPORT_DIR  = "/content/drive/MyDrive/l1_training_reports"
TEXT_ONLY_P = "/content/drive/MyDrive/text_only_logreg.joblib"

# Sütunlar
LABEL_COL = "l1"
ID_COL    = "id"
IMG_COL   = "img_emb"   # 768-d
TTL_COL   = "title"     # 768-d (önceden embedlenmiş)

# Boyutlar / Hiperparametreler
EMB_DIM   = 768
FUSED_DIM = 512
MIN_SAMPLES_PER_CLASS = 30

VAL_RATIO = 0.2
N_SPLITS  = int(round(1/VAL_RATIO))  # 5
SEED      = 42

# A100 batch
BATCH_TRAIN = 256
BATCH_VAL   = 512
EPOCHS      = 40
MAX_LR_NEW  = 1.5e-4   # fusion+head
WEIGHT_DECAY= 1e-3

# Loss
LABEL_SMOOTHING = 0.05
FOCAL_ALPHA     = 1.0
FOCAL_GAMMA     = 1.5

# ArcFace
ARCFACE_S = 30.0
ARCFACE_M = 0.30

# Ensemble weights
W_TEXT   = 0.6
W_FUSION = 0.4

os.makedirs(REPORT_DIR, exist_ok=True)

# -------------- UTILS --------------
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def _norm_pathlike_id(s):
    s = str(s)
    if "/" in s:  s = s.split("/")[-1]
    if "\\" in s: s = s.split("\\")[-1]
    s = re.sub(r"\.\w+$", "", s)
    return s

def group_key_from_id(raw_id):
    rid = _norm_pathlike_id(raw_id)
    if "_" in rid: return rid.split("_")[0]
    if "-" in rid: return rid.split("-")[0]
    return rid

def coerce_vec(x, name, idx):
    arr = np.asarray(x, dtype=np.float32)
    if arr.ndim != 1 or arr.shape[0] != EMB_DIM:
        raise ValueError(f"[Row {idx}] {name} shape {arr.shape}, expected ({EMB_DIM},)")
    return arr

def summarize_split(y_tr, y_va, le):
    def topbot(y):
        c = Counter(y)
        big5 = sorted(c.items(), key=lambda z: z[1], reverse=True)[:5]
        small5 = sorted(c.items(), key=lambda z: z[1])[:5]
        return big5, small5
    btr, strr = topbot(y_tr)
    bva, sva  = topbot(y_va)
    print(f"- Train sınıf sayısı: {len(set(y_tr))} | Val sınıf sayısı: {len(set(y_va))}")
    print("- Train en büyük 5:", [(int(k), v, le.classes_[int(k)]) for k, v in btr])
    print("- Val   en büyük 5:", [(int(k), v, le.classes_[int(k)]) for k, v in bva])
    print("- Train en küçük 5:", [(int(k), v, le.classes_[int(k)]) for k, v in strr])
    print("- Val   en küçük 5:", [(int(k), v, le.classes_[int(k)]) for k, v in sva])

# -------------- DATASET --------------
class EmbeddedEcomDataset(Dataset):
    def __init__(self, X_img, X_ttl, y):
        self.X_img = X_img.astype(np.float32)
        self.X_ttl = X_ttl.astype(np.float32)
        self.y     = y.astype(np.int64)

    def __len__(self): return self.X_img.shape[0]

    def __getitem__(self, idx):
        return {
            "image_features": torch.from_numpy(self.X_img[idx]),
            "title_features": torch.from_numpy(self.X_ttl[idx]),
            "labels": torch.tensor(self.y[idx], dtype=torch.long)
        }

# -------------- MODEL --------------
class ClassTokenAttention(nn.Module):
    def __init__(self, feature_dim, num_classes):
        super().__init__()
        self.feature_dim = feature_dim
        self.class_prototypes = nn.Parameter(torch.randn(num_classes, feature_dim))
        self.q = nn.Linear(feature_dim, feature_dim)
        self.k = nn.Linear(feature_dim, feature_dim)
        self.v = nn.Linear(feature_dim, feature_dim)
        self.drop = nn.Dropout(0.1)
        self.out = nn.Linear(feature_dim, feature_dim)
        nn.init.xavier_uniform_(self.class_prototypes)

    def forward(self, x):
        # x: [B, D]
        q = self.q(x)                                # [B, D]
        k = self.k(self.class_prototypes)            # [C, D]
        v = self.v(self.class_prototypes)            # [C, D]
        att = torch.matmul(q, k.t()) / (x.shape[1] ** 0.5)  # [B, C]
        w = F.softmax(att, dim=1)
        w = self.drop(w)
        attended = torch.matmul(w, v)                # [B, D]
        out = self.out(attended + x)                 # residual
        return out, w

class ArcMarginProduct(nn.Module):
    """
    ArcFace head: training'de margin uygular; eval'de labels=None ile saf cosine döner.
    """
    def __init__(self, in_dim, num_classes, s=30.0, m=0.30, easy_margin=False):
        super().__init__()
        self.s = s
        self.m = m
        self.easy_margin = easy_margin
        self.weight = nn.Parameter(torch.randn(num_classes, in_dim))
        nn.init.xavier_uniform_(self.weight)
        # trig sabitler
        self.cos_m = float(np.cos(m))
        self.sin_m = float(np.sin(m))
        self.th = float(np.cos(np.pi - m))
        self.mm = float(np.sin(np.pi - m) * m)

    def forward(self, x, labels=None):
        # x: [B, D]
        x = F.normalize(x, dim=1)
        W = F.normalize(self.weight, dim=1)
        cosine = torch.matmul(x, W.t())                # [B, C]
        if labels is None:
            return self.s * cosine                     # eval: margin yok

        sine = torch.sqrt(torch.clamp(1.0 - cosine**2, min=0.0))
        phi = cosine * self.cos_m - sine * self.sin_m  # cos(theta+m)
        if not self.easy_margin:
            phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1,1), 1.0)
        logits = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        logits *= self.s
        return logits

class MultiModalL1(nn.Module):
    """
    Gating (metin ağırlıklı): z = α*title + (1-α)*image  → fusion(768→512) → attn → ArcFace
    """
    def __init__(self, num_classes):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(EMB_DIM*2, 128),
            nn.GELU(),
            nn.Linear(128, 1)   # sigmoid scalar α
        )
        self.fusion = nn.Sequential(
            nn.Linear(EMB_DIM, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(1024, FUSED_DIM),
            nn.LayerNorm(FUSED_DIM),
            nn.GELU(),
            nn.Dropout(0.2),
        )
        self.attn = ClassTokenAttention(FUSED_DIM, num_classes)
        self.head = ArcMarginProduct(FUSED_DIM, num_classes, s=ARCFACE_S, m=ARCFACE_M)

        # init
        for m in self.fusion.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.LayerNorm):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, img_feats, ttl_feats, labels=None):
        img_feats = F.normalize(img_feats, dim=-1)
        ttl_feats = F.normalize(ttl_feats, dim=-1)
        concat = torch.cat([img_feats, ttl_feats], dim=1)
        alpha = torch.sigmoid(self.gate(concat))          # [B,1]
        z = alpha * ttl_feats + (1 - alpha) * img_feats   # [B,768]
        x = self.fusion(z)                                 # [B,512]
        x, attw = self.attn(x)                             # [B,512], [B,C]
        logits = self.head(x, labels)                      # train: labels var, val: None
        return logits, attw, alpha.squeeze(1)

# -------------- LOSSES --------------
class LabelSmoothingFocalLoss(nn.Module):
    def __init__(self, num_classes, smoothing=0.05, alpha=1.0, gamma=1.5, weight=None):
        super().__init__()
        self.num_classes = num_classes
        self.smoothing = smoothing
        self.alpha = alpha
        self.gamma = gamma
        self.register_buffer("weight", weight if weight is not None else None)

    def forward(self, inputs, targets):
        # label smoothing
        with torch.no_grad():
            confidence = 1.0 - self.smoothing
            smooth_neg = self.smoothing / (self.num_classes - 1)
            smoothed = torch.full_like(inputs, smooth_neg)
            smoothed.scatter_(1, targets.unsqueeze(1), confidence)

        logp = F.log_softmax(inputs, dim=1)
        pt = torch.exp(logp.gather(1, targets.unsqueeze(1)).squeeze(1))  # [B]
        focal = self.alpha * (1 - pt) ** self.gamma

        loss = -(smoothed * logp).sum(dim=1) * focal  # [B]
        if self.weight is not None:
            loss = loss * self.weight[targets]
        return loss.mean()

# -------------- TRAINER --------------
class Trainer:
    def __init__(self, model, device, train_loader, val_loader, num_classes, class_weights, label_classes):
        self.model = model.to(device)
        self.device = device
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.num_classes = num_classes
        self.label_classes = label_classes

        cw = torch.tensor(class_weights, dtype=torch.float32, device=device) if class_weights is not None else None
        self.criterion = LabelSmoothingFocalLoss(num_classes, LABEL_SMOOTHING, FOCAL_ALPHA, FOCAL_GAMMA, cw)

        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=MAX_LR_NEW, weight_decay=WEIGHT_DECAY)
        self.scheduler = torch.optim.lr_scheduler.OneCycleLR(
            self.optimizer,
            max_lr=MAX_LR_NEW,
            epochs=EPOCHS,
            steps_per_epoch=len(self.train_loader),
            pct_start=0.1,
            anneal_strategy='cos',
            div_factor=10,
            final_div_factor=100
        )

        self.scaler = torch.cuda.amp.GradScaler()
        self.best_val = 0.0

    def train_epoch(self):
        self.model.train()
        tot_loss, correct, total = 0.0, 0, 0
        alpha_meter = 0.0
        pbar = tqdm(self.train_loader, desc="Train")
        for batch in pbar:
            xi = batch["image_features"].to(self.device, non_blocking=True)
            xt = batch["title_features"].to(self.device, non_blocking=True)
            y  = batch["labels"].to(self.device, non_blocking=True)

            self.optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast():
                logits, _, alpha = self.model(xi, xt, y)   # training: labels var (ArcFace margin aktif)
                loss = self.criterion(logits, y)

            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.scaler.step(self.optimizer)
            self.scaler.update()
            self.scheduler.step()

            tot_loss += loss.item()
            pred = logits.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
            alpha_meter += alpha.mean().item()

            pbar.set_postfix(loss=f"{loss.item():.4f}",
                             acc=f"{100*correct/total:.2f}%",
                             alpha=f"{alpha_meter/(total//y.size(0) + 1e-9):.2f}")  # kaba izleme
        return tot_loss/len(self.train_loader), 100*correct/total

    @torch.no_grad()
    def validate(self):
        self.model.eval()
        tot_loss = 0.0
        all_logits, all_labels = [], []
        alpha_meter = 0.0
        pbar = tqdm(self.val_loader, desc="Val")
        for batch in pbar:
            xi = batch["image_features"].to(self.device, non_blocking=True)
            xt = batch["title_features"].to(self.device, non_blocking=True)
            y  = batch["labels"].to(self.device, non_blocking=True)
            with torch.cuda.amp.autocast():
                logits, _, alpha = self.model(xi, xt, labels=None)  # eval: labels=None (margin yok)
                loss = self.criterion(logits, y)
            tot_loss += loss.item()
            all_logits.append(logits.float().cpu().numpy())
            all_labels.append(y.cpu().numpy())
            alpha_meter += alpha.mean().item()

        all_logits = np.concatenate(all_logits, 0)
        all_labels = np.concatenate(all_labels, 0)
        top1 = accuracy_score(all_labels, all_logits.argmax(1))*100
        top5 = top1 if self.num_classes < 5 else top_k_accuracy_score(all_labels, all_logits, k=5)*100
        return tot_loss/len(self.val_loader), top1, top5, all_logits, all_labels, (alpha_meter/len(self.val_loader))

    def save_ckpt(self, path, epoch, val_top1, val_top5):
        torch.save({
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "epoch": epoch,
            "val_acc": val_top1,
            "val_top5": val_top5,
            "label_classes": list(self.label_classes)
        }, path)

    def _write_reports(self, y_true, y_pred):
        # sınıf isimleriyle classification report
        try:
            report = classification_report(
                y_true, y_pred,
                target_names=[str(x) for x in self.label_classes],
                digits=4, zero_division=0
            )
            with open(os.path.join(REPORT_DIR, "classification_report.txt"), "w", encoding="utf-8") as f:
                f.write(report)
            print("📄 classification_report.txt yazıldı.")

            cm = confusion_matrix(y_true, y_pred)
            np.save(os.path.join(REPORT_DIR, "confusion_matrix.npy"), cm)
            print("📊 confusion_matrix.npy yazıldı.")
        except Exception as e:
            print("Rapor yazımında hata:", e)

    def fit(self, epochs=EPOCHS):
        print(f"Starting training for {epochs} epochs")
        best = 0.0
        patience, max_patience = 0, 10
        for ep in range(1, epochs+1):
            print(f"\nEpoch {ep}/{epochs}")
            tr_loss, tr_acc = self.train_epoch()
            va_loss, va_top1, va_top5, logits, labels, alpha_val = self.validate()

            print(f"Train Loss: {tr_loss:.4f} | Train Acc: {tr_acc:.2f}%")
            print(f"Val   Loss: {va_loss:.4f} | Val Top-1: {va_top1:.2f}% | Val Top-5: {va_top5:.2f}% | α(mean)={alpha_val:.2f}")

            if va_top1 > best:
                best = va_top1
                patience = 0
                self.save_ckpt(CKPT_PATH, ep, va_top1, va_top5)
                print(f"✅ New best checkpoint saved to: {CKPT_PATH} (Top-1 {va_top1:.2f}%)")
                # Rapor + logits kaydet
                preds = logits.argmax(1)
                self._write_reports(labels, preds)
                np.save(os.path.join(REPORT_DIR, "fusion_logits_val.npy"), logits)
                np.save(os.path.join(REPORT_DIR, "fusion_labels_val.npy"), labels)
                print("💾 fusion_logits_val.npy & fusion_labels_val.npy yazıldı.")
            else:
                patience += 1
                print(f"No improvement. Patience {patience}/10")

            if patience >= max_patience:
                print("⏹️ Early stopping.")
                break

            if va_top1 >= 99.0:
                print("🎯 TARGET ACHIEVED ≥99%")
                break
        print(f"\nBest Val Top-1: {best:.2f}%")
        return best

# -------------- DATA PREP --------------
def load_and_prepare():
    print("="*70)
    print("1) Veri Yükleme ve Kolon Kontrolü")
    print("="*70)
    if not Path(FILE_PATH).exists():
        raise FileNotFoundError(FILE_PATH)
    df = pd.read_parquet(FILE_PATH)
    print(f"- Dosya: {FILE_PATH}")
    print(f"- Satır: {len(df):,}")
    for c in [ID_COL, IMG_COL, TTL_COL, LABEL_COL]:
        if c not in df.columns:
            raise KeyError(f"Eksik sütun: {c}")
    print("- Sütunlar OK:", [ID_COL, IMG_COL, TTL_COL, LABEL_COL])

    print("\n" + "="*70)
    print("2) NaN / Embed Şekil Kontrolü")
    print("="*70)
    before = len(df)
    df = df.dropna(subset=[IMG_COL, TTL_COL, LABEL_COL])
    print(f"- NaN temizliği: {before - len(df)} satır çıkarıldı")
    bad = []
    for i, (ie, te) in enumerate(zip(df[IMG_COL].values, df[TTL_COL].values)):
        try:
            _ = coerce_vec(ie, IMG_COL, i)
            _ = coerce_vec(te, TTL_COL, i)
        except Exception:
            bad.append(i)
    if bad:
        print(f"- Embed shape hatalı: {len(bad)} satır çıkarılıyor")
        df = df.drop(df.index[bad]).reset_index(drop=True)
    print(f"- Kalan satır: {len(df):,}")

    print("\n" + "="*70)
    print("3) L1 Sınıf Dağılımı (Ham)")
    print("="*70)
    vc = df[LABEL_COL].value_counts()
    print(f"- Sınıf sayısı: {vc.shape[0]} | Toplam: {vc.sum():,}")
    print("- En büyük 10:", vc.head(10).to_dict())
    print("- En küçük 10:", vc.tail(10).to_dict())

    print("\n" + "="*70)
    print("4) Min Örnek Eşiği")
    print("="*70)
    keep = vc[vc >= MIN_SAMPLES_PER_CLASS].index
    removed = set(vc.index) - set(keep)
    df = df[df[LABEL_COL].isin(keep)].reset_index(drop=True)
    print(f"- Eşik: {MIN_SAMPLES_PER_CLASS}")
    print(f"- Çıkarılan sınıf: {len(removed)} -> örn: {list(sorted(removed))[:10]}")
    print(f"- Kalan satır: {len(df):,} | Kalan sınıf: {df[LABEL_COL].nunique()}")

    print("\n" + "="*70)
    print("5) Grup Anahtarı (Leakage Önleme)")
    print("="*70)
    df["_group_key"] = df[ID_COL].map(group_key_from_id)
    gp_counts = df["_group_key"].value_counts()
    quant = gp_counts.quantile([0, .25, .5, .75, .9, .95, 1.0]).to_dict()
    print(f"- Grup sayısı: {gp_counts.shape[0]}")
    print("- Grup boyutu quantile'ları:", {k: int(v) for k, v in quant.items()})
    if gp_counts.max() == 1:
        print("⚠️  Tüm gruplar tekil görünüyor (yine de SGKF güvenli).")

    print("\n" + "="*70)
    print("6) Tek LabelEncoder ve Embedding Matrisleri")
    print("="*70)
    le = LabelEncoder()
    le.fit(sorted(df[LABEL_COL].unique()))
    y_all = le.transform(df[LABEL_COL])
    print(f"- Sınıf sayısı (LE): {len(le.classes_)}")
    print(f"- İlk 10 sınıf: {list(le.classes_)[:10]}")

    X_img = np.stack([coerce_vec(v, IMG_COL, i) for i, v in enumerate(df[IMG_COL].values)], axis=0)
    X_ttl = np.stack([coerce_vec(v, TTL_COL, i) for i, v in enumerate(df[TTL_COL].values)], axis=0)
    print(f"- X_img shape   : {X_img.shape}")
    print(f"- X_title shape : {X_ttl.shape}")
    print(f"- y_all shape   : {y_all.shape}")

    print("\n" + "="*70)
    print("7) StratifiedGroupKFold Split")
    print("="*70)
    sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    splits = list(sgkf.split(np.zeros(len(df)), y_all, df["_group_key"]))
    train_idx, val_idx = splits[0]
    print(f"- Train idx: {len(train_idx):,} | Val idx: {len(val_idx):,}")

    X_img_tr, X_img_va = X_img[train_idx], X_img[val_idx]
    X_ttl_tr, X_ttl_va = X_ttl[train_idx], X_ttl[val_idx]
    y_tr, y_va = y_all[train_idx], y_all[val_idx]

    summarize_split(y_tr, y_va, le)

    # class weights
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(len(le.classes_)),
        y=y_tr
    ).astype(np.float32)
    print(f"- Class weights: min={class_weights.min():.4f} max={class_weights.max():.4f} mean={class_weights.mean():.4f}")

    return (X_img_tr, X_ttl_tr, y_tr), (X_img_va, X_ttl_va, y_va), le, class_weights

# -------------- TEXT-ONLY + ENSEMBLE --------------
def train_text_only_and_ensemble(Xt_tr, Xt_va, y_tr, y_va):
    print("\n" + "="*70)
    print("Text-only (LogisticRegression) + Ensemble")
    print("="*70)

    scaler = StandardScaler(with_mean=True, with_std=True)
    Xt_tr_s = scaler.fit_transform(Xt_tr)
    Xt_va_s = scaler.transform(Xt_va)

    clf = LogisticRegression(
        multi_class="multinomial", solver="lbfgs",
        max_iter=2000, n_jobs=-1, C=4.0, class_weight="balanced"
    )
    clf.fit(Xt_tr_s, y_tr)
    proba_text = clf.predict_proba(Xt_va_s)
    pred_text = proba_text.argmax(1)
    acc_text = accuracy_score(y_va, pred_text)*100
    print(f"- Text-only Val Acc: {acc_text:.2f}%")

    joblib.dump({"scaler": scaler, "clf": clf}, TEXT_ONLY_P)
    print(f"💾 Saved text-only model to {TEXT_ONLY_P}")

    # Fusion logits yükle
    f_logits_p = os.path.join(REPORT_DIR, "fusion_logits_val.npy")
    if not os.path.exists(f_logits_p):
        print("⚠️ fusion logits bulunamadı; ensemble atlanıyor.")
        return None
    logits_fusion = np.load(f_logits_p)
    # softmax
    logits_fusion = logits_fusion - logits_fusion.max(1, keepdims=True)
    proba_fusion = np.exp(logits_fusion); proba_fusion /= proba_fusion.sum(1, keepdims=True)

    proba_ens = W_TEXT*proba_text + W_FUSION*proba_fusion
    pred_ens = proba_ens.argmax(1)
    acc_ens = accuracy_score(y_va, pred_ens)*100
    print(f"- Ensemble Val Acc (w_text={W_TEXT}): {acc_ens:.2f}%")
    return {"acc_text": acc_text, "acc_ens": acc_ens}

# -------------- MAIN --------------
def main():
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("="*70)
    print("L1 Training (Pre-embedded img + title) — A100 Optimized")
    print("Device:", device)
    print("="*70)

    (Xi_tr, Xt_tr, y_tr), (Xi_va, Xt_va, y_va), le, class_weights = load_and_prepare()
    num_classes = len(le.classes_)

    # Datasets
    train_ds = EmbeddedEcomDataset(Xi_tr, Xt_tr, y_tr)
    val_ds   = EmbeddedEcomDataset(Xi_va, Xt_va, y_va)

    # Weighted sampler
    binc = np.bincount(y_tr, minlength=num_classes)
    binc[binc == 0] = 1
    weights = 1.0 / binc
    sample_weights = weights[y_tr]
    sampler = WeightedRandomSampler(torch.DoubleTensor(sample_weights),
                                    num_samples=len(sample_weights),
                                    replacement=True)

    # Loaders
    train_loader = DataLoader(train_ds, batch_size=BATCH_TRAIN, sampler=sampler,
                              num_workers=4, pin_memory=True, persistent_workers=False)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_VAL, shuffle=False,
                              num_workers=4, pin_memory=True, persistent_workers=False)

    # Model
    model = MultiModalL1(num_classes=num_classes)

    # Trainer
    trainer = Trainer(model, device, train_loader, val_loader, num_classes, class_weights, le.classes_)
    best = trainer.fit(EPOCHS)

    print("\nFINAL RESULT (Best Val Top-1): {:.2f}%".format(best))
    if best < 99.0:
        print("Still {:.2f}% away from 99% target".format(99.0 - best))

    # Text-only + ENSEMBLE
    metrics = train_text_only_and_ensemble(Xt_tr, Xt_va, y_tr, y_va)
    if metrics is not None:
        with open(os.path.join(REPORT_DIR, "ensemble_summary.json"), "w", encoding="utf-8") as f:
            json.dump(metrics, f, ensure_ascii=False, indent=2)
        print("📌 ensemble_summary.json kaydedildi.")

if __name__ == "__main__":
    main()


L1 Training (Pre-embedded img + title) — A100 Optimized
Device: cuda
1) Veri Yükleme ve Kolon Kontrolü
- Dosya: /content/drive/MyDrive/final_master_df_title_embedded.parquet
- Satır: 72,221
- Sütunlar OK: ['id', 'img_emb', 'title', 'l1']

2) NaN / Embed Şekil Kontrolü
- NaN temizliği: 0 satır çıkarıldı
- Kalan satır: 72,221

3) L1 Sınıf Dağılımı (Ham)
- Sınıf sayısı: 84 | Toplam: 72,221
- En büyük 10: {'Yapı Market & Bahçe': 9869, 'Evcil Hayvan Ürünleri': 4416, 'Süpermarket': 2858, 'Kadın Giyim & Aksesuar': 2803, 'Bilgisayar': 2573, 'Kırtasiye & Ofis': 2548, 'Aksesuar & Tuning': 2307, 'Mutfak Gereçleri': 2299, 'Kitap': 2289, 'Bireysel & Takım Sporları': 1960}
- En küçük 10: {'Bebek Oyuncakları': 99, 'Dijital Kodlar & Ürünler': 93, 'Oto Koltuğu & Ana Kucağı': 80, 'Güneş & Bronzluk Ürünleri': 72, 'Bebek Bezi & Islak Mendil': 59, 'Yürüteç & Yürüme Yardımcıları': 40, 'Ev Kozmetiği': 40, 'Yaşam ve Etkinlik': 21, 'El İşi Ürünleri': 20, 'Kampanya Kozmetik': 3}

4) Min Örnek Eşiği
- Eşik: 30
-

Val: 100%|██████████| 29/29 [00:00<00:00, 98.77it/s] 


Train Loss: 39.8613 | Train Acc: 0.45%
Val   Loss: 4.3876 | Val Top-1: 3.17% | Val Top-5: 8.76% | α(mean)=0.46
✅ New best checkpoint saved to: /content/drive/MyDrive/best_l1_multimodal_arcface.pth (Top-1 3.17%)
📄 classification_report.txt yazıldı.
📊 confusion_matrix.npy yazıldı.
💾 fusion_logits_val.npy & fusion_labels_val.npy yazıldı.

Epoch 2/40


Val: 100%|██████████| 29/29 [00:00<00:00, 94.25it/s] 


Train Loss: 24.7175 | Train Acc: 7.92%
Val   Loss: 3.5005 | Val Top-1: 9.51% | Val Top-5: 25.29% | α(mean)=0.58
✅ New best checkpoint saved to: /content/drive/MyDrive/best_l1_multimodal_arcface.pth (Top-1 9.51%)
📄 classification_report.txt yazıldı.
📊 confusion_matrix.npy yazıldı.
💾 fusion_logits_val.npy & fusion_labels_val.npy yazıldı.

Epoch 3/40


...

Train Loss: 1.2216 | Train Acc: 75.47%
Val   Loss: 1.6099 | Val Top-1: 63.45% | Val Top-5: 86.13% | α(mean)=0.64
No improvement. Patience 1/10

Epoch 39/40


Val: 100%|██████████| 29/29 [00:00<00:00, 104.82it/s]


Train Loss: 1.2418 | Train Acc: 75.37%
Val   Loss: 1.6097 | Val Top-1: 63.46% | Val Top-5: 86.16% | α(mean)=0.64
No improvement. Patience 2/10

Epoch 40/40


Val: 100%|██████████| 29/29 [00:00<00:00, 98.58it/s] 


Train Loss: 1.2159 | Train Acc: 75.60%
Val   Loss: 1.6104 | Val Top-1: 63.43% | Val Top-5: 86.19% | α(mean)=0.64
No improvement. Patience 3/10

Best Val Top-1: 63.48%

FINAL RESULT (Best Val Top-1): 63.48%
Still 35.52% away from 99% target

Text-only (LogisticRegression) + Ensemble
- Text-only Val Acc: 68.00%
💾 Saved text-only model to /content/drive/MyDrive/text_only_logreg.joblib
- Ensemble Val Acc (w_text=0.6): 70.76%
📌 ensemble_summary.json kaydedildi.


In [ ]:
!pip install xgboost -q

In [ ]:
# ============================================================
# L1 Baselines (Pre-embedded img/title) — A100 Ready
# - SGKF split (leakage guard)
# - Logistic Regression (text-only, img-only)
# - XGBoost (GPU, fused 1536-d)
# - Weighted Ensemble grid-search (text/img/xgb)
# - Saves models + reports
# ============================================================
import os, re, json, warnings, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
try:
    from sklearn.metrics import top_k_accuracy_score
    HAVE_TOPK = True
except Exception:
    HAVE_TOPK = False

from sklearn.linear_model import LogisticRegression
import joblib

# XGBoost (GPU)
try:
    import xgboost as xgb
    HAVE_XGB = True
except Exception as e:
    print("⚠️ xgboost import edilemedi. GPU XGBoost atlanacak. Hata:", e)
    HAVE_XGB = False

# ---------------- CONFIG ----------------
FILE_PATH   = "/content/drive/MyDrive/final_master_df_title_embedded.parquet"
OUT_DIR     = "/content/drive/MyDrive/l1_baselines"
os.makedirs(OUT_DIR, exist_ok=True)

ID_COL      = "id"
IMG_COL     = "img_emb"   # 768-dim
TTL_COL     = "title"     # 768-dim (embedlenmiş)
LABEL_COL   = "l1"

EMB_DIM     = 768
FUSED_DIM   = 1536        # 768 + 768
MIN_SAMPLES = 30
SEED        = 42

VAL_RATIO   = 0.2
N_SPLITS    = int(round(1/VAL_RATIO))  # 5

# Logistic Regression (multinomial)
LR_C        = 4.0
LR_MAXITER  = 3000
LR_SOLVER   = "lbfgs"  # (n_jobs paramı bu solver için dikkate alınmaz, sorun değil)

# XGBoost (GPU) params — iyi genel amaçlı başlangıç
XGB_PARAMS = dict(
    objective="multi:softprob",
    tree_method="gpu_hist",
    predictor="gpu_predictor",
    max_depth=8,
    learning_rate=0.10,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_alpha=0.0,
    reg_lambda=1.0,
    min_child_weight=2,
    n_estimators=2000,          # early_stopping ile durur
    eval_metric="mlogloss",
    random_state=SEED,
)

# -------------- UTILS --------------
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)

def _norm_pathlike_id(s):
    s = str(s)
    if "/" in s:  s = s.split("/")[-1]
    if "\\" in s: s = s.split("\\")[-1]
    s = re.sub(r"\.\w+$", "", s)
    return s

def group_key_from_id(raw_id):
    rid = _norm_pathlike_id(raw_id)
    if "_" in rid: return rid.split("_")[0]
    if "-" in rid: return rid.split("-")[0]
    return rid

def coerce_vec(x, name, idx):
    arr = np.asarray(x, dtype=np.float32)
    if arr.ndim != 1 or arr.shape[0] != EMB_DIM:
        raise ValueError(f"[Row {idx}] {name} shape {arr.shape}, expected ({EMB_DIM},)")
    return arr

def print_head_tail(d, k=10):
    head = dict(list(d.items())[:k])
    tail = dict(list(d.items())[-k:])
    return head, tail

def topk_acc(y_true, proba, k=5):
    if not HAVE_TOPK or proba.shape[1] < k:
        return accuracy_score(y_true, proba.argmax(1)) * 100.0
    return top_k_accuracy_score(y_true, proba, k=k) * 100.0

# -------------- DATA PREP --------------
def load_and_prepare():
    print("="*70)
    print("L1 Baseline Training (Pre-embedded img + title) — A100")
    print("="*70)
    if not Path(FILE_PATH).exists():
        raise FileNotFoundError(FILE_PATH)
    df = pd.read_parquet(FILE_PATH)
    print(f"Dosya: {FILE_PATH}")
    print(f"Satır: {len(df):,}")
    print(f"Sütun kontrol: {[ID_COL, IMG_COL, TTL_COL, LABEL_COL]}")

    # NaN temizliği
    before = len(df)
    df = df.dropna(subset=[IMG_COL, TTL_COL, LABEL_COL])
    print(f"NaN temizliği: {before - len(df)} çıkarıldı | Kalan: {len(df):,}")

    # embed shape doğrulama (hız için ilk 2000'i sıkı kontrol, sonra sampling)
    bad_idx = []
    check_n = min(2000, len(df))
    for i in range(check_n):
        try:
            _ = coerce_vec(df.iloc[i][IMG_COL], IMG_COL, i)
            _ = coerce_vec(df.iloc[i][TTL_COL], TTL_COL, i)
        except Exception:
            bad_idx.append(i)
    if bad_idx:
        print(f"⚠️ İlk {check_n} kayıtta shape hatası: {len(bad_idx)} — satırlar atılıyor.")
        df = df.drop(df.index[bad_idx]).reset_index(drop=True)
    print(f"Kontrol sonrası satır: {len(df):,}")

    # Ham dağılım
    vc = df[LABEL_COL].value_counts()
    print(f"Sınıf sayısı (ham): {vc.shape[0]} | Toplam: {vc.sum():,}")
    h, t = print_head_tail(vc.to_dict(), 10)
    print("En büyük 10:", h)
    print("En küçük 10:", t)

    # Min örnek eşiği
    keep = vc[vc >= MIN_SAMPLES].index
    removed = sorted(set(vc.index) - set(keep))
    df = df[df[LABEL_COL].isin(keep)].reset_index(drop=True)
    print(f"Eşik: {MIN_SAMPLES} | Çıkarılan sınıf: {len(removed)} | Kalan: {len(df):,} satır, {df[LABEL_COL].nunique()} sınıf")

    # Grup anahtarı
    df["_group_key"] = df[ID_COL].map(group_key_from_id)
    gp_counts = df["_group_key"].value_counts()
    quant = gp_counts.quantile([0, .25, .5, .75, .9, .95, 1.0]).to_dict()
    print("Grup sayısı:", gp_counts.shape[0])
    print("Grup boyutu quantile'ları:", {k: int(v) for k, v in quant.items()})

    # LabelEncoder + matrisler
    le = LabelEncoder()
    le.fit(sorted(df[LABEL_COL].unique()))
    y_all = le.transform(df[LABEL_COL])
    X_img = np.stack([coerce_vec(v, IMG_COL, i) for i, v in enumerate(df[IMG_COL].values)], axis=0)
    X_ttl = np.stack([coerce_vec(v, TTL_COL, i) for i, v in enumerate(df[TTL_COL].values)], axis=0)
    print(f"X_img: {X_img.shape}, X_title: {X_ttl.shape}, y: {y_all.shape}, classes: {len(le.classes_)}")

    # SGKF split
    sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    splits = list(sgkf.split(np.zeros(len(df)), y_all, df["_group_key"]))
    train_idx, val_idx = splits[0]
    Xi_tr, Xi_va = X_img[train_idx], X_img[val_idx]
    Xt_tr, Xt_va = X_ttl[train_idx], X_ttl[val_idx]
    y_tr,  y_va  = y_all[train_idx], y_all[val_idx]

    # özet
    def _tops(y):
        c = Counter(y)
        big5 = sorted(c.items(), key=lambda z: z[1], reverse=True)[:5]
        small5 = sorted(c.items(), key=lambda z: z[1])[:5]
        return big5, small5
    btr, strr = _tops(y_tr); bva, sva = _tops(y_va)
    print(f"Train/Val: {len(y_tr):,}/{len(y_va):,} | Sınıf sayısı: {len(set(y_tr))}/{len(set(y_va))}")
    print("Train en büyük 5:", [(int(k), v, le.classes_[int(k)]) for k, v in btr])
    print("Val   en büyük 5:", [(int(k), v, le.classes_[int(k)]) for k, v in bva])
    print("Train en küçük 5:", [(int(k), v, le.classes_[int(k)]) for k, v in strr])
    print("Val   en küçük 5:", [(int(k), v, le.classes_[int(k)]) for k, v in sva])

    # class weights (LR ve XGB sample_weight için)
    classes = np.arange(len(le.classes_))
    from sklearn.utils.class_weight import compute_class_weight
    cls_w = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr).astype(np.float32)
    print("Class weights: min={:.4f}, max={:.4f}, mean={:.4f}".format(cls_w.min(), cls_w.max(), cls_w.mean()))

    meta = {
        "file": FILE_PATH,
        "train_size": int(len(y_tr)),
        "val_size": int(len(y_va)),
        "num_classes": int(len(le.classes_)),
        "min_samples": MIN_SAMPLES,
        "classes": list(le.classes_),
    }
    with open(os.path.join(OUT_DIR, "meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    return (Xi_tr, Xt_tr, y_tr), (Xi_va, Xt_va, y_va), le, cls_w

# -------------- MODELS --------------
def train_lr_text_only(Xt_tr, Xt_va, y_tr, y_va, out_dir=OUT_DIR):
    print("\n" + "="*70)
    print("Logistic Regression — TEXT ONLY (title 768-d)")
    print("="*70)
    scaler = StandardScaler(with_mean=True, with_std=True)
    Xt_tr_s = scaler.fit_transform(Xt_tr)
    Xt_va_s = scaler.transform(Xt_va)

    clf = LogisticRegression(
        multi_class="multinomial",
        solver=LR_SOLVER,
        C=LR_C,
        max_iter=LR_MAXITER,
        class_weight="balanced",
        n_jobs=-1,               # lbfgs'te dikkate alınmaz; sorun değil
        verbose=0
    )
    clf.fit(Xt_tr_s, y_tr)
    proba = clf.predict_proba(Xt_va_s)
    acc1  = accuracy_score(y_va, proba.argmax(1))*100.0
    acc5  = topk_acc(y_va, proba, k=5)
    print(f"Text-only Val Top-1: {acc1:.2f}% | Top-5: {acc5:.2f}%")

    joblib.dump({"scaler": scaler, "clf": clf}, os.path.join(out_dir, "lr_text_only.joblib"))
    np.save(os.path.join(out_dir, "proba_text_val.npy"), proba)
    return proba, acc1, acc5

def train_lr_img_only(Xi_tr, Xi_va, y_tr, y_va, out_dir=OUT_DIR):
    print("\n" + "="*70)
    print("Logistic Regression — IMAGE ONLY (image 768-d)")
    print("="*70)
    scaler = StandardScaler(with_mean=True, with_std=True)
    Xi_tr_s = scaler.fit_transform(Xi_tr)
    Xi_va_s = scaler.transform(Xi_va)

    clf = LogisticRegression(
        multi_class="multinomial",
        solver=LR_SOLVER,
        C=LR_C,
        max_iter=LR_MAXITER,
        class_weight="balanced",
        n_jobs=-1,
        verbose=0
    )
    clf.fit(Xi_tr_s, y_tr)
    proba = clf.predict_proba(Xi_va_s)
    acc1  = accuracy_score(y_va, proba.argmax(1))*100.0
    acc5  = topk_acc(y_va, proba, k=5)
    print(f"Image-only Val Top-1: {acc1:.2f}% | Top-5: {acc5:.2f}%")

    joblib.dump({"scaler": scaler, "clf": clf}, os.path.join(out_dir, "lr_image_only.joblib"))
    np.save(os.path.join(out_dir, "proba_image_val.npy"), proba)
    return proba, acc1, acc5

def train_xgb_fused(Xi_tr, Xt_tr, Xi_va, Xt_va, y_tr, y_va, cls_w, num_classes, out_dir=OUT_DIR):
    if not HAVE_XGB:
        print("\n❌ XGBoost kullanılamıyor; bu adım atlandı.")
        return None, None, None, None
    print("\n" + "="*70)
    print("XGBoost (GPU) — FUSED (img+title = 1536-d)")
    print("="*70)
    Xtr = np.concatenate([Xi_tr, Xt_tr], axis=1).astype(np.float32)  # (N,1536)
    Xva = np.concatenate([Xi_va, Xt_va], axis=1).astype(np.float32)

    # per-sample weights (class imbalance)
    sample_w = cls_w[y_tr]

    xgb_clf = xgb.XGBClassifier(
        num_class=num_classes,
        **XGB_PARAMS,
        early_stopping_rounds=100
    )
    xgb_clf.fit(
        Xtr, y_tr,
        sample_weight=sample_w,
        eval_set=[(Xva, y_va)],
        verbose=False
    )

    # predict_proba
    proba = xgb_clf.predict_proba(Xva)
    acc1  = accuracy_score(y_va, proba.argmax(1))*100.0
    acc5  = topk_acc(y_va, proba, k=5)
    print(f"XGB Fused Val Top-1: {acc1:.2f}% | Top-5: {acc5:.2f}%")
    xgb_clf.save_model(os.path.join(out_dir, "xgb_fused.json"))
    np.save(os.path.join(out_dir, "proba_xgb_val.npy"), proba)
    return proba, acc1, acc5, xgb_clf

# -------------- ENSEMBLE --------------
def ensemble_grid_search(y_va, proba_dict, step=0.1, out_dir=OUT_DIR):
    """
    proba_dict: {"text": proba_ndarray, "img": proba_ndarray, "xgb": proba_ndarray (optional)}
    Grid: w in {0, step, 2*step, ..., 1}, sum(w)=1
    """
    keys = list(proba_dict.keys())
    P = [proba_dict[k] for k in keys]
    K = len(keys)

    def gen_weights(k, s):
        # K=2 or 3 için küçük grid
        vals = np.arange(0.0, 1.0 + 1e-9, s)
        if k == 2:
            for a in vals:
                b = 1.0 - a
                yield (a, b)
        elif k == 3:
            for a in vals:
                for b in vals:
                    c = 1.0 - a - b
                    if c < -1e-9: continue
                    if c > 1.0 + 1e-9: continue
                    yield (a, b, c)
        else:
            # Daha fazla model eklersen gerektiğinde genişletilir.
            raise NotImplementedError("Grid sadece 2-3 model için ayarlandı.")

    best = {"weights": None, "acc1": -1, "acc5": -1}
    tried = 0
    for ws in gen_weights(K, step):
        # combine
        ens = np.zeros_like(P[0])
        for w, p in zip(ws, P):
            ens += w * p
        acc1 = accuracy_score(y_va, ens.argmax(1))*100.0
        acc5 = topk_acc(y_va, ens, k=5)
        tried += 1
        if acc1 > best["acc1"]:
            best = {"weights": dict(zip(keys, [float(x) for x in ws])), "acc1": acc1, "acc5": acc5}
    print(f"Ensemble grid tried: {tried} combos")
    print("Best weights:", best["weights"])
    print(f"Best Ensemble Val Top-1: {best['acc1']:.2f}% | Top-5: {best['acc5']:.2f}%")

    with open(os.path.join(out_dir, "ensemble_best.json"), "w", encoding="utf-8") as f:
        json.dump(best, f, ensure_ascii=False, indent=2)
    return best

# -------------- REPORTS --------------
def write_reports(y_true, proba, class_names, prefix, out_dir=OUT_DIR):
    pred = proba.argmax(1)
    rep = classification_report(y_true, pred, target_names=[str(x) for x in class_names],
                                digits=4, zero_division=0)
    with open(os.path.join(out_dir, f"{prefix}_classification_report.txt"), "w", encoding="utf-8") as f:
        f.write(rep)
    cm = confusion_matrix(y_true, pred)
    np.save(os.path.join(out_dir, f"{prefix}_confusion_matrix.npy"), cm)

# -------------- MAIN --------------
def main():
    set_seed(SEED)
    (Xi_tr, Xt_tr, y_tr), (Xi_va, Xt_va, y_va), le, cls_w = load_and_prepare()
    num_classes = len(le.classes_)

    # # --- Train Text-only LR ---
    # proba_text, acc1_t, acc5_t = train_lr_text_only(Xt_tr, Xt_va, y_tr, y_va)
    # write_reports(y_va, proba_text, le.classes_, "lr_text_only", OUT_DIR)

    # # --- Train Image-only LR ---
    # proba_img, acc1_i, acc5_i = train_lr_img_only(Xi_tr, Xi_va, y_tr, y_va)
    # write_reports(y_va, proba_img, le.classes_, "lr_image_only", OUT_DIR)

    # --- Train XGB (GPU) on Fused ---
    proba_xgb, acc1_x, acc5_x, xgb_model = train_xgb_fused(
        Xi_tr, Xt_tr, Xi_va, Xt_va, y_tr, y_va, cls_w, num_classes, OUT_DIR
    )

    # --- Ensemble(s) ---
    # 2'li ve 3'lü senaryolara bak: (text+img), (text+xgb), (img+xgb), (text+img+xgb)
    results = {
        "text": {"acc1": acc1_t, "acc5": acc5_t},
        "img":  {"acc1": acc1_i, "acc5": acc5_i},
    }
    proba_dict = {"text": proba_text, "img": proba_img}

    if proba_xgb is not None:
        results["xgb"] = {"acc1": acc1_x, "acc5": acc5_x}
        proba_dict3 = {"text": proba_text, "img": proba_img, "xgb": proba_xgb}

    print("\n" + "="*70)
    print("ENSEMBLE GRID SEARCH (step=0.1)")
    print("="*70)

    best2 = ensemble_grid_search(y_va, proba_dict, step=0.1, out_dir=OUT_DIR)
    if proba_xgb is not None:
        best3 = ensemble_grid_search(y_va, proba_dict3, step=0.1, out_dir=OUT_DIR)
    else:
        best3 = None

    summary = {
        "single_models": results,
        "best_ensemble_2": best2,
        "best_ensemble_3": best3,
        "notes": "Proba dosyaları ve raporlar OUT_DIR altında. xgb yoksa best_ensemble_3 None olur."
    }
    with open(os.path.join(OUT_DIR, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("\n" + "="*70)
    print("ÖZET")
    print("="*70)
    print(json.dumps(summary, ensure_ascii=False, indent=2))

if __name__ == "__main__":
    main()


L1 Baseline Training (Pre-embedded img + title) — A100
Dosya: /content/drive/MyDrive/final_master_df_title_embedded.parquet
Satır: 72,221
Sütun kontrol: ['id', 'img_emb', 'title', 'l1']
NaN temizliği: 0 çıkarıldı | Kalan: 72,221
Kontrol sonrası satır: 72,221
Sınıf sayısı (ham): 84 | Toplam: 72,221
En büyük 10: {'Yapı Market & Bahçe': 9869, 'Evcil Hayvan Ürünleri': 4416, 'Süpermarket': 2858, 'Kadın Giyim & Aksesuar': 2803, 'Bilgisayar': 2573, 'Kırtasiye & Ofis': 2548, 'Aksesuar & Tuning': 2307, 'Mutfak Gereçleri': 2299, 'Kitap': 2289, 'Bireysel & Takım Sporları': 1960}
En küçük 10: {'Bebek Oyuncakları': 99, 'Dijital Kodlar & Ürünler': 93, 'Oto Koltuğu & Ana Kucağı': 80, 'Güneş & Bronzluk Ürünleri': 72, 'Bebek Bezi & Islak Mendil': 59, 'Yürüteç & Yürüme Yardımcıları': 40, 'Ev Kozmetiği': 40, 'Yaşam ve Etkinlik': 21, 'El İşi Ürünleri': 20, 'Kampanya Kozmetik': 3}
Eşik: 30 | Çıkarılan sınıf: 3 | Kalan: 72,177 satır, 81 sınıf
Grup sayısı: 72171
Grup boyutu quantile'ları: {0.0: 1, 0.25: 1, 0

NameError: name 'acc1_t' is not defined